We have a few extra instances that we are including in our analysis. We must split these as well.

In [ ]:
import json
import random
from pathlib import Path
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes

In [ ]:
CSV_BASE = Path('').resolve().parent / 'csv'

In [ ]:
df = pd.read_csv(CSV_BASE / 'SWE_Bench_M_Intersection.csv')
df.head()

In [ ]:
df = df.drop(columns=['GUIRepair + o4 mini', 'SWE-agent Multimodal + GPT 4o'])
df = df.rename(
    columns={
        'OpenHands + Claude-Sonnet 4': 'openhands',
        'GUIRepair + o3': 'guirepair',
        'Refact.ai Agent': 'refact',
    }
)

In [ ]:
df.head()

In [ ]:
def info(df: pd.DataFrame):
    print(df['openhands'].value_counts().to_string())
    print('')
    print(df['guirepair'].value_counts().to_string())
    print('')
    print(df['refact'].value_counts().to_string())


info(df)

In [ ]:
df = df.replace({'Yes': True, 'No': False})
df['openhands'] = df['openhands'].astype(bool)
df['guirepair'] = df['guirepair'].astype(bool)
df['refact'] = df['refact'].astype(bool)

In [ ]:
info(df)

In [ ]:
df_at_least_1_success = df.query(
    'openhands == True or guirepair == True or refact == True'
).copy()
df_at_least_1_success.shape

In [ ]:
df_all_success = df.query(
    'openhands == True and guirepair == True and refact == True'
).copy()
df_all_success.shape

In [ ]:
df_running = pd.read_csv(CSV_BASE / 'running_log.csv')
instance_ids_current = set(df_running['instance_id'].unique().tolist())
instance_ids_at_least_1_success = set(
    df_at_least_1_success['instance_id'].unique().tolist()
)

In [ ]:
df_running.head()

In [ ]:
instance_ids_to_add = instance_ids_at_least_1_success.difference(
    instance_ids_current
)
print(f'We need to add {len(instance_ids_to_add)} instances:')
print(instance_ids_to_add)

In [ ]:
df_unresolved = pd.read_csv(CSV_BASE / 'unresolved.csv')
mask = ~df_unresolved['instance_id'].isin(
    instance_ids_current.union(instance_ids_at_least_1_success).union(
        instance_ids_to_add
    )
)
df_unresolved = df_unresolved[mask].copy()

# Get the proportions in df_unresolved
proportions = df_unresolved.groupby(['commit_year', 'repo']).size() / len(
    df_unresolved
)
proportions_map = proportions.to_dict()

N = len(instance_ids_to_add)

# Sample N rows maintaining those proportions
parts: list[pd.DataFrame] = []
for name, group in df_unresolved.groupby(['commit_year', 'repo']):
    n = max(1, int(N * proportions_map[name]))
    parts.append(group.sample(n=n, random_state=42))
df_unresolved_subset = pd.concat(parts)

# Adjust to exactly N if rounding caused drift
if len(df_unresolved_subset) > N:
    df_unresolved_subset = df_unresolved_subset.sample(n=N, random_state=42)
elif len(df_unresolved_subset) < N:
    remaining = df_unresolved.drop(df_unresolved_subset.index)
    gap = remaining.sample(n=N - len(df_unresolved_subset), random_state=42)
    df_unresolved_subset = pd.concat([df_unresolved_subset, gap])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes = cast(list[Axes], axes)

# Commit year distribution
commit_counts = pd.DataFrame(
    {
        'Unresolved': df_unresolved.groupby('commit_year').size().sort_index(),
        'Subset': df_unresolved_subset.groupby('commit_year')
        .size()
        .sort_index(),
    }
).fillna(0)
commit_counts.index = commit_counts.index.astype(int)

x = np.arange(len(commit_counts))
axes[0].plot(
    commit_counts.index,
    commit_counts['Unresolved'],
    label='Unresolved',
    marker='o',
)
axes[0].plot(
    commit_counts.index, commit_counts['Subset'], label='Subset', marker='o'
)
axes[0].set_title('Commit Date Distribution')
axes[0].set_xlabel('Year')
axes[0].legend()

# Repo distribution
repo_counts = pd.DataFrame(
    {
        'Unresolved': df_unresolved['repo'].value_counts(),
        'Subset': df_unresolved_subset['repo'].value_counts(),
    }
).fillna(0)

x = np.arange(len(repo_counts))
axes[1].plot(
    repo_counts.index,
    repo_counts['Unresolved'],
    label='Unresolved',
    marker='o',
)
axes[1].plot(
    repo_counts.index, repo_counts['Subset'], label='Subset', marker='x'
)
axes[1].set_xticklabels(repo_counts.index, rotation=90)
axes[1].set_title('Repo Distribution')
axes[1].set_xlabel('Repo')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
unresolved_instance_ids_to_add = set(
    df_unresolved_subset['instance_id'].unique().tolist()
)

df_full = pd.read_csv(CSV_BASE / 'full-with-expanded-dates.csv')
df_to_add = df_full[
    df_full['instance_id'].isin(
        instance_ids_to_add.union(unresolved_instance_ids_to_add)
    )
].copy()

df_to_add = df_to_add.reindex(columns=df_running.columns)
df_to_add = df_to_add.drop(columns='Unnamed: 0')
df_to_add.head()

In [ ]:
df_to_add.shape

In [ ]:
def normalize_image_assets(val: str) -> list[str] | None:
    if pd.isna(val):
        return None
    val = str(val).strip()
    try:
        parsed = json.loads(val)
        urls = [url for urls in parsed.values() for url in urls]
    except (json.JSONDecodeError, AttributeError):
        urls = [val]

    return urls


df_to_add['image_assets'] = df_to_add['image_assets'].apply(
    normalize_image_assets
)
df_to_add = df_to_add.explode('image_assets', ignore_index=True)
df_to_add.head()

In [ ]:
idx = df_to_add.index.tolist()

# Shuffle and split exactly 50/50
random.shuffle(idx)
kumar = set(idx[: len(idx) // 2])
parsa = set(idx[len(idx) // 2 :])

random.shuffle(idx)
pairing_a = set(idx[: len(idx) // 2])
pairing_b = set(idx[len(idx) // 2 :])

# For pairing B, split tan/daniel exactly 50/50
pairing_b_list = list(pairing_b)
random.shuffle(pairing_b_list)
b_tan = set(pairing_b_list[: len(pairing_b_list) // 2])
b_daniel = set(pairing_b_list[len(pairing_b_list) // 2 :])

full_cols = list(df_to_add.columns)
keep_cols = [c for c in full_cols if c not in ['key', 'value']]
rows: list[pd.Series] = []

for idx, row in df_to_add.iterrows():
    base = row[keep_cols].copy()
    r1, r2 = base.copy(), base.copy()
    if idx in pairing_a:
        r1['name'] = 'kumar'
        r2['name'] = 'yaseen'
    else:
        r1['name'] = 'parsa'
        r2['name'] = 'tan' if idx in b_tan else 'daniel'
    rows.extend([r1, r2])

df_new_rows = pd.DataFrame(rows, columns=keep_cols)
df_final = pd.concat([df_running, df_new_rows], ignore_index=True)
df_final.head()

In [ ]:
df_final.to_csv(CSV_BASE / 'running_log_updated.csv', index=False)

In [ ]:
df_new_rows.to_csv(
    CSV_BASE / 'additional_rows_for_running_log.csv', index=False
)
df_new_rows.head()